# placement: is each piece centered on its square? (full design, perception)

README "Perception and rules": *verify physical placement: piece centered on its square
within tolerance*. This is separate from rule legality (`board_assembly.ipynb`) and from the
engine comparison (`board_sync.ipynb`). The robot needs it after its own moves (did the
gripper drop the piece off-center?) and before grasping (where exactly is the piece?).

Two parts:
1. **Detection**: estimate where a piece's base touches the board, in warp pixels.
   Use the base, not the silhouette center: tall pieces lean in perspective.
2. **Tolerance check**: convert that point to board meters with the warp geometry from
   `homography.ipynb`, subtract the square center, compare with a tolerance.

Milestone: full design (beyond the class project M1-M5).

**TODO**
- [ ] tolerance check: `placement_offset`, `check_placement`, `check_all`, `which_square`
- [ ] `estimate_piece_base`: choose a method (e.g. segment the piece against the known square
      color in the warped image and take the lowest blob point / fitted base ellipse; or a
      detector bbox bottom-center from `yolo.ipynb`)
- [ ] choose `tolerance_m` from the gripper's capture range (how far off-center can a piece
      be and still be grasped?), maybe separate x/y tolerances
- [ ] correct for the piece height: a point seen above the board plane maps to the wrong
      place under a board-plane homography
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import chess
import numpy as np

%run ./homography.ipynb

## Offsets and the tolerance check
Offset = (piece base in board meters) - (square center in board meters), so it is in
meters along the files (`du`, toward h) and ranks (`dv`, toward 8), independent of the warp
resolution and view.

In [ ]:
DEFAULT_TOLERANCE_FRAC = 0.2   # TODO: placeholder, 20% of a square (5 mm on the 25 mm board)


@dataclass
class PlacementCheck:
    square: str
    offset_m: tuple[float, float]   # (du, dv) in meters
    distance_m: float
    ok: bool


def placement_offset(sq: int, base_warp_xy, geom: BoardGeometry, warp) -> np.ndarray:
    """(du, dv) meters from the center of square sq to a point given in warp pixels."""
    # Design: warp_to_board(point) - square_center_board(sq). Working in board meters makes
    # the tolerance independent of warp resolution and view.
    # TODO: implement
    raise NotImplementedError


def check_placement(sq: int, base_warp_xy, geom: BoardGeometry, warp,
                    tolerance_m: float | None = None) -> PlacementCheck:
    """Is the piece on square sq within tolerance_m (Euclidean) of the square center?

    tolerance_m None -> DEFAULT_TOLERANCE_FRAC * square.
    """
    # TODO: implement (placement_offset + np.hypot)
    raise NotImplementedError


def check_all(bases: dict[int, tuple[float, float]], geom: BoardGeometry, warp,
              tolerance_m: float | None = None) -> list[PlacementCheck]:
    """Check every occupied square (sorted by square index); bases maps square index ->
    base point in warp pixels."""
    # TODO: implement
    raise NotImplementedError


def which_square(base_warp_xy, geom: BoardGeometry, warp) -> int | None:
    """Square index containing the point, or None if it is off the playing area."""
    # Design: warp_to_board / square gives (file, rank) as floats; outside [0, 8) -> None;
    # else chess.square(int(file), int(rank)).
    # TODO: implement
    raise NotImplementedError

## Detection

In [ ]:
def estimate_piece_base(warped: np.ndarray, sq: int, geom: BoardGeometry, warp) -> tuple[float, float] | None:
    """Base point (x, y) in warp pixels of the piece on square sq, or None if empty.

    Inputs: the warped top-down BGR image (homography.ipynb: warp_board) and a square index.
    """
    # Design:
    # - Look in a window around the square (the piece may be off-center by up to ~half a square).
    # - Compare with the expected square color (light/dark) to get a piece mask.
    # - The base is the part of the mask closest to the camera (bottom of the black_bottom
    #   warp); fit a circle/ellipse to the bottom contour (cv2.findContours, cv2.fitEllipse).
    # - Validate in sim, where true positions are known.
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# With geom = load_board_geometry() and warp = SimpleNamespace(square_px=100, margin_px=100,
# view="black_bottom"):
# - the exact e4 center (board_to_warp(square_center_board(e4))) -> ok, distance ~0, square "e4".
# - 4 mm toward the h-file is within the default 5 mm; 6 mm toward rank 8 is not and its
#   offset_m == (0, 0.006).
# - which_square of that 6 mm point is still e4; which_square([5, 5]) is None (in the margin).
# - check_all({e4: 4 mm point, d4: 6 mm point}) returns d4 first: [False, True].
print("placement: self-check not written yet (see plan above)")